# Essencial do Scikit Learn

In [195]:
from pathlib import Path

DATA_PATH = "../dataset/raw/casas.csv"
COMPLEX_DATA_PATH = "../dataset/raw/housing.csv"
MODEL_PATH = "../models/essencial-do-scikit-learn.joblib"

## Fluxo de Treinamento e Predição

Antes de tudo, precisamos de uma base de dados. Criamos um DataFrame abaixo com alguns dados fictícios para usarmos no treinamento do modelo. Saiba desde já uma informação muito importante: os algoritmos conseguem interpretar APENAS números. Logo mais, apresentaremos uma forma de tornar dados em texto interpretáveis.

In [196]:
import pandas
from sklearn.linear_model import LinearRegression

casas = pandas.DataFrame(
    {
        "Area": [48, 60, 73],
        "Quartos": [3, 2, 4],
        "Banheiros": [1, 2, 3],
        "Valor": [240000, 320000, 460000],
    }
)

Ao treinar um modelo, sempre devemos passar duas variáveis como parâmetros: X e y. A variável X deve ser um DataFrame contendo todas as colunas que são necessárias e relevantes para predizer a variável y. Esta variável y, por sua vez, é um objeto do tipo Series do pandas. Por isso, ao selecionar os dados X, utilizamos colchetes duplos para extrair um DataFrame a partir do original. E ao selecionar os dados y, usamos apenas colchetes simples para retornar uma Series.

In [197]:
X = casas[["Area", "Quartos", "Banheiros"]]
y = casas["Valor"]

Para treinar o modelo, primeiro devemos instanciá-lo de acordo com o algoritmo que tivermos escolhido para melhor cumprir a tarefa. Logo depois, podemos chamar o método fit, que receberá os parâmetros X e y. Com isso, o algoritmo encontrará um padrão nos dados (caso exista) e será capaz de predizer novos resultados aplicando esse padrão aprendido a novos dados.

In [198]:
model = LinearRegression()

model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](3,)","[ 8092.82,17302.39, 188.54]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](3,)","['Area','Quartos','Banheiros']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-2.006e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


Para testarmos se o treinamento funcionou, podemos definir um DataFrame de entrada contendo valores para todos os campos selecionados em X. Com isso feito, já podemos chamar o método predict do modelo, passando como argumento os dados de entrada a serem utilizados na predição.

In [199]:
area = 50
bedrooms = 2
bathrooms = 2

input_dataframe = pandas.DataFrame({
    "Area": [area],
    "Quartos": [bedrooms],
    "Banheiros": [bathrooms]
})

prediction = model.predict(input_dataframe)

Os modelos que treinamos sempre podem processar mais de um registro por chamada. Se passarmos um DataFrame com todas as colunas de X para o método predict, ele retornará um array com os resultados para todas as linhas. Por isso, o retorno do predict é sempre um array da biblioteca numpy. Portanto, quando predizemos um valor y para uma única linha de X, acessamos o índice zero do resultado (prediction[0]) para obter o primeiro (e único) valor resultante.

In [200]:
estimated_value = prediction[0]

print(f"Valor estimado: R$ {estimated_value:.2f}")

Valor estimado: R$ 239071.79


## Separação de Dados de Treino e Teste

Quando trabalhamos em um projeto real, temos uma quantidade limitada de dados reais. Por isso, não podemos utilizar toda a base de dados apenas para o treinamento. Afinal, como vamos validar se o modelo consegui extrair um padrão se não tivermos dados para testar? Além disso, porque devemos separar os dados de treino e teste logo no início?

Devemos sempre separar os dados de treino e teste antes do treinamento do modelo pelo seguinte motivo: vamos preencher dados vazios, escalar dados numéricos e muitas outras operações. Essas operações geralmente utilizam os dados já existentes na base de dados. Não fazemos a separação depois dessas operações porque isso contaminaria os dados de treinamento e o modelo treinado teria uma performance maior do que a realidade, visto que logo após validaremos os resultados em um modelo que já viu dados um pouco mais similares aos dados de teste.

Para separar os dados, é muito simples. Apenas extraímos X e y da base de dados e passamos como parâmetros para a função train_test_split do scikit-learn. Além dos dados, podemos passar os parâmetros random_state e test_size. Por padrão, essa função gera um número aleatório como seed e esse número vai determinar quais linhas da base de dados serão selecionadas para treino ou teste. O parâmetro random_state vai fixar o número que especificarmos como seed. Dessa forma, o resultado da separação será sempre o mesmo. Já o test_size define a porção dos dados totais que será destinada aos testes. Um valor de 0.2 define que 20% dos dados terão essa finalidade.

A função train_test_split retorna 4 valores que representam os dados destinados a cada etapa. Primeiro o X de treino, depois o X de teste. Logo depois dos dados X, o y de treino e o y de teste. Em toda a próxima etapa de pré-processamento, utilizaremos apenas o X de treino e o y de treino.

In [201]:
from sklearn.model_selection import train_test_split

casas = pandas.read_csv(DATA_PATH)

X = casas[["Area", "Quartos", "Banheiros", "Tem_Garagem"]]
y = casas["Valor"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

## Pré-processamento

Abaixo, que o nosso arquivo CSV local com alguns dados de casas contém campos vazios. A base de dados em que trabalharemos, no mundo real terá diversos problemas como colunas mal nomeadas, duplicatas, valores faltando, valores fora de escala, colunas para informações irrelevantes, entre outros. Por isso, é necessário fazermos o pré-processamento dos dados para que o modelo possa ser treinado da melhor forma possível e dessa forma fazer as predições com a maior qualidade possível.

In [202]:
casas.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Area         94 non-null     float64
 1   Quartos      94 non-null     float64
 2   Banheiros    94 non-null     float64
 3   Tem_Garagem  100 non-null    str    
 4   Valor        100 non-null    int64  
dtypes: float64(3), int64(1), str(1)
memory usage: 4.0 KB


### Preenchendo valores nulos

Ao lidar com bases de dados do mundo real, é altamente provável que encontremos linhas com dados faltantes. Antes de entregarmos os dados para o algoritmo extrair padrões, precisamos formatar e tratar esses dados para que o algoritmo consiga compreendê-los. Uma parte essencial dessa preparação é o preenchimento (imputação) ou a remoção das linhas com dados faltantes.

#### SimpleImputer e método fit_transform

Para fazer a imputação (preenchimento com base em outros valores) dos dados faltantes, podemos criar um objeto da classe SimpleImputer. O construtor da classe SimpleImputer recebe um parâmetro strategy, que define como preencheremos os valores vazios. A estratégia "median" preenche os dados inserindo a mediana dos valores da coluna. Por exemplo, se a mediana do número de quartos for 3, todas as linhas em que o número de quartos era nulo agora receberão o valor 3.

Um ponto importante a lembrar é que o SimpleImputer realiza a imputação apenas em dados numéricos. Por esse motivo, separamos os dados em colunas numéricas (X_numerical) e colunas categóricas (X_categorical).

In [203]:
from sklearn.impute import SimpleImputer

X_numerical = X_train[["Area", "Quartos", "Banheiros"]]
X_categorical = X_train[["Tem_Garagem"]]
y = y_train

imputer = SimpleImputer(strategy="median")

Para preencher os dados faltantes, chamamos o método fit_transform do imputer, passando como parâmetro o DataFrame X_numerical. Com isso, todos os valores faltantes em X_numerical serão preenchidos com a mediana de suas respectivas colunas.

In [204]:
X_numerical_imputed = imputer.fit_transform(X_numerical)

print(X_numerical_imputed)

[[213.9    4.     2.  ]
 [184.8    3.     2.  ]
 [ 69.8    2.     1.  ]
 [ 66.8    1.     1.  ]
 [139.3    4.     1.  ]
 [ 82.     3.     2.  ]
 [ 93.8    1.     1.  ]
 [171.8    5.     3.  ]
 [ 68.     3.     2.  ]
 [180.     2.     1.  ]
 [105.25   2.     1.  ]
 [ 77.5    2.     1.  ]
 [ 92.     2.     1.  ]
 [196.2    3.     3.  ]
 [ 43.     2.     1.  ]
 [105.25   2.     1.  ]
 [159.9    2.     2.  ]
 [105.25   2.     1.  ]
 [ 74.4    3.     2.  ]
 [105.     4.     3.  ]
 [147.4    3.     3.  ]
 [ 79.8    3.     1.  ]
 [ 45.     1.     1.  ]
 [167.8    3.     2.  ]
 [ 52.3    2.     1.  ]
 [ 64.     2.     2.  ]
 [111.2    5.     3.  ]
 [ 69.8    1.     1.  ]
 [130.1    4.     2.  ]
 [ 58.     2.     1.  ]
 [148.6    3.     1.  ]
 [ 52.     2.     1.  ]
 [ 50.     3.     1.  ]
 [135.4    3.     1.  ]
 [110.     4.     3.  ]
 [ 76.9    3.     2.  ]
 [ 92.     3.     2.  ]
 [203.9    2.     1.  ]
 [ 74.6    3.     2.  ]
 [ 55.3    3.     3.  ]
 [185.9    3.     2.  ]
 [105.8    3.   

Podemos facilmente visualizar nosso array bidimensional em formato de DataFrame: basta usar a classe DataFrame do pandas, passando o array bidimensional e o parâmetro columns para nomear as colunas. Como a ordem das colunas nunca mudou e é preservada no atributo columns de X_numerical, tudo fica perfeitamente alinhado.

In [205]:
print(pandas.DataFrame(X_numerical_imputed, columns=X_numerical.columns))

     Area  Quartos  Banheiros
0   213.9      4.0        2.0
1   184.8      3.0        2.0
2    69.8      2.0        1.0
3    66.8      1.0        1.0
4   139.3      4.0        1.0
..    ...      ...        ...
75  120.3      4.0        3.0
76  214.6      3.0        1.0
77   95.0      3.0        2.0
78   77.4      4.0        2.0
79  150.1      1.0        1.0

[80 rows x 3 columns]


#### O que é um ndarray?

Um ndarray é um objeto da biblioteca numpy que representa um array de N dimensões (*n-dimensional array*). O retorno do imputer no nosso exemplo foi um ndarray de duas dimensões (matriz bidimensional). Sabemos disso porque ele se comporta como um array com outros arrays dentro. Para visualizarmos um ndarray como DataFrame, ele precisa obrigatoriamente ter duas dimensões.

O método reshape do ndarray altera a estrutura de um array de acordo com as dimensões que informamos. Quando o valor de uma dimensão recebe -1, indicamos que aquele tamanho deve ser calculado dinamicamente. Nesse primeiro exemplo abaixo, nosso array resultante terá uma única dimensão (array unidimensional plano).

In [206]:
import numpy

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

print(numpy.array(numbers).reshape(-1))

[ 1  2  3  4  5  6  7  8  9 10 11 12]


Neste outro exemplo, definimos que o array de resultado terá duas dimensões. A primeira dimensão terá uma quantidade dinâmica de itens (-1) e a segunda dimensão terá exatamente 1 item.

In [207]:
print(numpy.array(numbers).reshape(-1, 1))

[[ 1]
 [ 2]
 [ 3]
 [ 4]
 [ 5]
 [ 6]
 [ 7]
 [ 8]
 [ 9]
 [10]
 [11]
 [12]]


Passando outros valores, como no exemplo abaixo, definimos que a primeira dimensão terá tamanho 3 e a segunda dimensão terá tamanho 4 (uma matriz 3x4).

In [208]:
print(numpy.array(numbers).reshape(3, 4))

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]


Por fim, um último exemplo: aqui definimos que o array resultante terá 3 dimensões (2 x 2 x 3). Dessa forma, a camada externa terá 2 arrays. Cada um deles terá outros 2 arrays dentro de si. E cada um desses arrays internos terá 3 itens. Ao usar o reshape, a multiplicação de todas as dimensões especificadas deve resultar exatamente no número total de elementos do array original (neste exemplo, 2 x 2 x 3 = 12).

In [209]:
print(numpy.array(numbers).reshape(2, 2, 3))

[[[ 1  2  3]
  [ 4  5  6]]

 [[ 7  8  9]
  [10 11 12]]]


#### Dados nulos na série de resultado (alvo)

Na enorme maioria dos casos, quando há linhas em que a variável de resultado (y) possui valor nulo, simplesmente descartamos essas linhas. Não seria analiticamente correto preenchê-las, pois estaríamos treinando nosso modelo com base em dados artificiais (gabarito inventado).

### Escalando dados

No exemplo de casas que estamos usando, temos um problema de proporcionalidade: a área varia de 40 a 200, enquanto o número de quartos varia de 1 a 4 e o de banheiros também de 1 a 4. Isso faz com que o modelo fique excessivamente sensível a mudanças na área apenas pela grandeza dos números, atribuindo menos peso a atributos que são igualmente importantes.

Para equilibrar essa diferença, utilizamos um escalador (*scaler*). No exemplo abaixo, usamos a classe MinMaxScaler. O objeto MinMaxScaler possui o método fit_transform, que retorna os dados em uma nova escala. O MinMaxScaler define o menor valor de uma coluna como 0 e o maior como 1, distribuindo proporcionalmente todos os valores intermediários no intervalo entre 0 e 1.

In [210]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_numerical_scaled = scaler.fit_transform(X_numerical_imputed)

print(pandas.DataFrame(X_numerical_scaled, columns=X_numerical.columns))

        Area  Quartos  Banheiros
0   0.995921     0.75   0.333333
1   0.826340     0.50   0.333333
2   0.156177     0.25   0.000000
3   0.138695     0.00   0.000000
4   0.561189     0.75   0.000000
..       ...      ...        ...
75  0.450466     0.75   0.666667
76  1.000000     0.50   0.000000
77  0.303030     0.50   0.333333
78  0.200466     0.75   0.333333
79  0.624126     0.00   0.000000

[80 rows x 3 columns]


### Interpretando texto

Antes de continuarmos, é importante distinguir as colunas numéricas das colunas categóricas. Em termos simples, colunas numéricas contêm dados quantitativos contínuos ou discretos (como área e quantidade de quartos). Já as colunas categóricas representam categorias ou rótulos (como "Sim" e "Não", nomes de bairros, cidades, etc.).

No exemplo abaixo, instanciamos a classe OneHotEncoder e chamamos o método fit_transform. Por padrão, o encoder retorna uma matriz esparsa (*sparse matrix*), que é um formato compacto otimizado para economizar memória quando há muitos zeros. Definindo o parâmetro sparse_output=False, recebemos de volta um array tradicional do numpy (matriz densa).

Em resumo, o OneHotEncoder gera uma nova coluna para cada categoria possível da variável original. Para cada linha, apenas a coluna correspondente à categoria observada recebe o valor 1, enquanto as demais recebem o valor 0.

In [211]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
X_categorical_encoded = encoder.fit_transform(X_categorical)
categorical_columns = encoder.get_feature_names_out()

print(pandas.DataFrame(X_categorical_encoded, columns=categorical_columns))

    Tem_Garagem_Nao  Tem_Garagem_Sim
0               1.0              0.0
1               1.0              0.0
2               1.0              0.0
3               0.0              1.0
4               0.0              1.0
..              ...              ...
75              1.0              0.0
76              0.0              1.0
77              0.0              1.0
78              1.0              0.0
79              1.0              0.0

[80 rows x 2 columns]


Para terminar o pré-processamento manual, construímos DataFrames com as respectivas colunas para cada matriz processada e, por fim, juntamos tudo lado a lado utilizando a função concat do pandas com axis=1.

Com isso, nossos dados estão devidamente tratados e prontos para treinar o modelo.

In [212]:
dataframe_X_numerical = pandas.DataFrame(X_numerical_scaled, columns=X_numerical.columns)
dataframe_X_categorical = pandas.DataFrame(X_categorical_encoded, columns=categorical_columns)
X_processed = pandas.concat([dataframe_X_numerical, dataframe_X_categorical], axis=1)

display(X_processed)

,Area,Quartos,Banheiros,Tem_Garagem_Nao,Tem_Garagem_Sim
0,0.995921,0.75,0.333333,1.0,0.0
1,0.826340,0.50,0.333333,1.0,0.0
2,0.156177,0.25,0.000000,1.0,0.0
3,0.138695,0.00,0.000000,0.0,1.0
4,0.561189,0.75,0.000000,0.0,1.0
...,...,...,...,...,...
75,0.450466,0.75,0.666667,1.0,0.0
76,1.000000,0.50,0.000000,0.0,1.0
77,0.303030,0.50,0.333333,0.0,1.0
78,0.200466,0.75,0.333333,1.0,0.0


### Pipelines

#### Configurando uma Pipeline

Foi um tanto trabalhoso construir o fluxo parte por parte e passar os dados de uma parte para outra passo a passo, não foi? Bom, o scikit-learn, para facilitar a configuração desse fluxo, nos oferece as pipelines.

Se lembra que quando fizemos manualmente, passamos as colunas numéricas por um SimpleImputer e depois por um MinMaxScaler? Faremos exatamente a mesma coisa agora, porém, configurando uma pipeline. Primeiro, é ideal definirmos um array com todas as nossas colunas numéricas.

A classe Pipeline do scikit-learn recebe um parâmetro steps que deve ser um array de tuplas. Cada tupla terá como primeiro item o nome do passo e a ferramenta que utilizaremos para realizá-lo. Abaixo, definimos um passo chamado "imputer" que vai realizar a imputação dos dados utilizando a mediana. Depois do imputer, definimos um passo "scaler" que vai utilizar o MinMaxScaler. Dessa forma, não precisamos extrair manualmente o resultado do SimpleImputer e passá-lo para o MinMaxScaler. A pipeline já faz isso automaticamente.

In [213]:
from sklearn.pipeline import Pipeline

numerical_columns = ["Area", "Quartos", "Banheiros"]

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")), 
        ("scaler", MinMaxScaler())
    ]
)

Agora, podemos fazer exatamente a mesma coisa para as colunas categóricas. Aqui, criamos outra pipeline que vai processá-las. Essa pipeline, nesse caso, passará as colunas categóricas apenas pelo OneHotEncoder. Um parâmetro que não mostramos ainda é o handle_unknown. Esse parâmetro definirá como o modelo deve reagir caso receba um valor diferente de todos os valores que havia no treinamento. Nesse caso, definimos como "ignore". Então todas as colunas que o OneHotEncoder tiver gerado terão valor 0 para o registro com categoria diferente.

In [214]:
categorical_columns = ["Tem_Garagem"]

categorical_pipeline = Pipeline(
    steps=[
        ("onehotencoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

Para juntar as duas pipelines em uma coisa só, utilizamos a classe ColumnTransformer. Essa classe recebe um parâmetro transformers que vai receber um array de tuplas. Cada tupla terá três itens. Primeiro o nome do transformador, depois a pipeline ou ferramenta que será utilizada e por fim, quais colunas serão utilizadas no transformador. O primeiro transformador se chama "numerical" e vai aplicar todas as regras da pipeline numérica nas colunas numéricas que definimos antes no array numerical_columns. Já o segundo transformador se chama "categorical" e vai aplicar todas as regras da pipeline categórica nas colunas categóricas que definimos antes no array categorical_columns.

In [215]:

from sklearn.compose import ColumnTransformer

column_transformer = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ]
)

Terminando, podemos chamar o método fit_transform do column_transformer passando como argumento o dataframe. Essa chamada já realizará todo o pré-processamento dos dados que configuramos com as pipelines. Nesse exemplo, o processed_data será um array bidimensional que representa linhas e colunas. Portanto, podemos criar um dataframe a partir desse resultado. O objeto do tipo ColumnTransformer tem um método get_feature_names_out. Esse método retornará uma lista de todos os nomes de colunas que haviam ao final do pré-processamento dos dados. Os nomes das colunas incluem o nome do processador pelo qual elas passaram para facilitar encontrarmos algum eventual erro na configuração das pipelines.

In [216]:
processed_data = column_transformer.fit_transform(X_train)
processed_dataframe = pandas.DataFrame(
    processed_data, columns=column_transformer.get_feature_names_out()
)

display(processed_dataframe)

,numerical__Area,numerical__Quartos,numerical__Banheiros,categorical__Tem_Garagem_Nao,categorical__Tem_Garagem_Sim
0,0.995921,0.75,0.333333,1.0,0.0
1,0.826340,0.50,0.333333,1.0,0.0
2,0.156177,0.25,0.000000,1.0,0.0
3,0.138695,0.00,0.000000,0.0,1.0
4,0.561189,0.75,0.000000,0.0,1.0
...,...,...,...,...,...
75,0.450466,0.75,0.666667,1.0,0.0
76,1.000000,0.50,0.000000,0.0,1.0
77,0.303030,0.50,0.333333,0.0,1.0
78,0.200466,0.75,0.333333,1.0,0.0


A título de comparação, este é o fluxo realizado manualmente

In [217]:
X_numerical = X_train[["Area", "Quartos", "Banheiros"]]
X_categorical = X_train[["Tem_Garagem"]]

imputer = SimpleImputer(strategy="median")
scaler = MinMaxScaler()
encoder = OneHotEncoder(sparse_output=False)

X_numerical_imputed = imputer.fit_transform(X_numerical)
X_numerical_scaled = scaler.fit_transform(X_numerical_imputed)
numerical_columns = imputer.get_feature_names_out()

X_categorical_encoded = encoder.fit_transform(X_categorical)
categorical_columns = encoder.get_feature_names_out()

X_numerical_dataframe = pandas.DataFrame(X_numerical_scaled, columns=numerical_columns)
X_categorical_dataframe = pandas.DataFrame(
    X_categorical_encoded, columns=categorical_columns
)

print(pandas.concat([X_numerical_dataframe, X_categorical_dataframe], axis=1))

        Area  Quartos  Banheiros  Tem_Garagem_Nao  Tem_Garagem_Sim
0   0.995921     0.75   0.333333              1.0              0.0
1   0.826340     0.50   0.333333              1.0              0.0
2   0.156177     0.25   0.000000              1.0              0.0
3   0.138695     0.00   0.000000              0.0              1.0
4   0.561189     0.75   0.000000              0.0              1.0
..       ...      ...        ...              ...              ...
75  0.450466     0.75   0.666667              1.0              0.0
76  1.000000     0.50   0.000000              0.0              1.0
77  0.303030     0.50   0.333333              0.0              1.0
78  0.200466     0.75   0.333333              1.0              0.0
79  0.624126     0.00   0.000000              1.0              0.0

[80 rows x 5 columns]


E este é o mesmo fluxo organizado com pipelines. Lembrem-se: o mais importante não é um código ter menos linhas, e sim que facilite o reuso, que seja mais legível e que seja mais compreensível.

In [218]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

numerical_columns = ["Area", "Quartos", "Banheiros"]
categorical_columns = ["Tem_Garagem"]

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")), 
        ("scaler", MinMaxScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("onehotencoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

column_transformer = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ]
)

processed_data = column_transformer.fit_transform(X_train)
processed_dataframe = pandas.DataFrame(
    processed_data, columns=column_transformer.get_feature_names_out()
)

display(processed_dataframe)

,numerical__Area,numerical__Quartos,numerical__Banheiros,categorical__Tem_Garagem_Nao,categorical__Tem_Garagem_Sim
0,0.995921,0.75,0.333333,1.0,0.0
1,0.826340,0.50,0.333333,1.0,0.0
2,0.156177,0.25,0.000000,1.0,0.0
3,0.138695,0.00,0.000000,0.0,1.0
4,0.561189,0.75,0.000000,0.0,1.0
...,...,...,...,...,...
75,0.450466,0.75,0.666667,1.0,0.0
76,1.000000,0.50,0.000000,0.0,1.0
77,0.303030,0.50,0.333333,0.0,1.0
78,0.200466,0.75,0.333333,1.0,0.0


#### Incluindo o processamento do modelo

Para incluir o modelo ao fluxo configurado com pipelines, basta criamos outro objeto da classe Pipeline que terá dois passos: o passo de pré-processamento com o nosso objeto column_transformer e o passo do modelo que passará os dados do pré-processamento para o modelo.

In [219]:
full_pipeline = Pipeline(
    steps=[
        ("preprocessing", column_transformer),
        ("model", LinearRegression())
    ]
)

Agora, finalmente podemos passar nossos dados de treino diretamente para a pipeline completa. Dessa forma, podemos chamar o método fit da pipeline e este já aplicará todo o pré-processamento nos dados e treinará o modelo. 

In [220]:
full_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['Area','Quartos','Banheiros','Tem_Garagem']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenate

Para fazer uma predição, a mesma lógica. Basta chamar o método predict e passar como parâmetro os dados necessários para a predição (X), da mesma forma que fazíamos quando chamávamos o método direto do objeto do modelo.

In [221]:
prediction = full_pipeline.predict(pandas.DataFrame({
    "Area": [56],
    "Quartos": [2],
    "Banheiros": [2],
    "Tem_Garagem": ["Sim"]
}))

print(f"Valor estimado: R$ {prediction[0]:.2f}")

Valor estimado: R$ 328240.80


#### Importância de usar pipelines

Se lembra que lá no início falamos sobre a separação de dados de treino e dados de teste? Após todo o pré-processamento manual, concorda que ficaria um tanto difícil replicar todo o pré-processamento nos dados de treino manualmente? Agora, como configuramos uma pipeline que realiza todo esse fluxo, abstraímos esse pré-processamento e não precisaremos configurá-lo de novo do zero.

Note também como se torna extremamente fácil trocar o algoritmo de treinamento. Se quisermos utilizar outros tipos de algoritmos de regressão, basta editarmos uma única linha de código na pipeline.

## Métricas para Avaliação

Para algoritmos de regressão, temos algumas funções que avaliam o modelo treinado. A primeira delas é a função mean_absolute_error (MAE). Sempre que avaliarmos o modelo, precisamos dos dados y de teste e dos dados y preditos. Para gerar os dados y preditos, basta utilizar o método predict do modelo ou pipeline passando como parâmetro os dados X de teste. A função mean_absolute_error recebe um parâmetro y_true que é o gabarito e um parâmetro y_pred que é a resposta.

Então, como essa função calcula o resultado? A função compara o resultado de cada predição e extrai a diferença absolute entre ela e o gabarito. Se o modelo estimou 250.000 e o gabarito era 275.000, a diferença absoluta para essa predição foi de 25.000. No final, são somadas as diferenças absolutas para cada predição do modelo e por fim essa soma é dividida pela quantidade de resultados preditos. A pergunta que essa função responde, em resumo, é: Em média, por quanto o modelo errou?

In [222]:
from sklearn.metrics import mean_absolute_error

y_predicted = full_pipeline.predict(X_test)

print(mean_absolute_error(y_true=y_test, y_pred=y_predicted))

22406.76578526062


Outra função que podemos utilizar é a mean_squared_error (MSE). Ela faz quase exatamente o que a MAE faz. A única diferença é que quando a diferença absoluta é calculada, ela também é elevada ao quadrado antes de fazer o somatório para todas as predições. Essa métrica penaliza fortemente erros altos.

In [223]:
from sklearn.metrics import mean_squared_error

print(mean_squared_error(y_true=y_test, y_pred=y_predicted))

749097306.0033448


A terceira principal função é a root_mean_squared_error (RMSE). Esta função também recebe y_true e y_pred igual a anterior. A diferença está em como o cálculo é feito. Estes são os passos:

1. Calcula a diferença para cada predição
2. Eleva cada diferença ao quadrado
3. Calcula a média entre todos os quadrados
4. Tira a raíz quadrada da média

Por elevar a diferença de cada predição ao quadrado, o modelo é mais penalizado conforme as predições sejam mais distantes do gabarito em comparação ao MAE (mean_absolute_error). O resultado dessa função será sempre igual ou maior do que na função MAE. Além disso, o resultado dessa função é exatamente a raíz quadrada do resultado da função MSE.

In [224]:
from sklearn.metrics import root_mean_squared_error

print(root_mean_squared_error(y_true=y_test, y_pred=y_predicted))

27369.642051063525


A quarta principal função é a r2_score. Estes são os passos do cálculo:

1. Calcula o MSE para o modelo treinado
2. Calcula o MSE se o modelo sempre respondesse exatamente a média de todos os valores do y de teste
3. Divide o MSE do modelo treinado pelo MSE da média
4. Subtrai o resultado de 1 (1 - MSE_modelo / MSE_media)

Naturalmente, o resultado da função estará entre 0 e 1. Porém, é completamente possível que o resultado seja negativo. Quando isso acontece, o modelo treinado teve uma performance pior do que se tivéssemos chutado sempre a média de todos os valores. Em resumo, essa função responde a seguinte pergunta: O modelo elimina que porcentagem de todos os erros caso tivéssemos chutado sempre a média? 

Uma outra forma de entender o resultado é interpretando o quanto do padrão o modelo consegue explicar.

In [225]:
from sklearn.metrics import r2_score

print(r2_score(y_true=y_test, y_pred=y_predicted))

0.9871485532312392


Existem muitas outras funções que extraem métricas com base nos resultados de predição de modelos de regressão. Porém, os principais e mais utilizados são apenas estes 4.

### Avaliação em fatias

Imagine o seguinte problema: você faz a separação dos dados de treino e dados de teste, treina o modelo e testa. O modelo teve um resultado bom. Você separa os dados mais uma vez, só que de uma forma diferente e agora o modelo não teve um resultado tão bom. Isso significa que por coincidência, você pode ter acabado separando uma fatia mais fácil de prever dos dados como dados de teste.

A avaliação em fatias é de extrema importância quando temos poucos dados (< 100.000 registros>). Para fazer a avaliação em fatias, utilizamos a função cross_val_score, passando como parâmetros a pipeline ou modelo, X de treino e y de treino, um parâmetro cv que define o número de fatias (dobras) e um parâmetro scoring que define que tipo de métrica será gerada. A função por fim retorna um ndarray com a pontuação para cada dobra.

Mas afinal, o que a função faz? Ela divide os dados em 5 partes iguais (dobras) por padrão ou o número que especificarmos pelo parâmetro cv. A cada dobra, a função utiliza as outras 4 partes para treinar o modelo e a quinta parte para testar. Um exemplo de fluxo: Considere os nomes das dobras: A, B, C, D, E

- Treina com B, C, D, E e testa com A
- Treina com A, C, D, E e testa com B
- Treina com A, B, D, E e testa com C
- Treina com A, B, C, E e testa com D
- Treina com A, B, C, D e testa com E

Ou seja, isso torna muito, muito difícil termos um resultado bom por coincidência.

In [226]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring="r2")

print(scores)

[0.90583083 0.98026425 0.98574739 0.78316028 0.93540441]


Como a função retorna um ndarray, podemos extrair a média entre as pontuações utilizando o método mean. Também é útil verificarmos o desvio padrão com o método std.

In [227]:
print(f"R² médio: {scores.mean():.3f} +- {scores.std():.3f}")

R² médio: 0.918 +- 0.074


## Exportando e Importando a Pipeline Completa

Agora que já configuramos toda a pipeline de processamento dos dados integrada ao modelo, avaliamos o modelo e ele teve um bom desempenho, podemos exportá-lo como um arquivo para utilizar em um projeto ou solução. Para isso, basta importarmos a biblioteca joblib e utilizar o método dump dela, passando como parâmetros a pipeline ou modelo que queremos exportar e onde queremos que o arquivo seja salvo.

In [228]:
import joblib

joblib.dump(full_pipeline, MODEL_PATH)

['../models/essencial-do-scikit-learn.joblib']

Para importar o modelo ou pipeline, também é muito simples. Basta utilizar o método load do joblib passando como argumento o caminho onde o arquivo está salvo.

In [229]:
imported_pipeline = joblib.load(MODEL_PATH)

## Essencial de Feature Engineering

### Preparativos

Nessa parte, estamos apenas renomeando as colunas para facilitar o entendimento durante os estudos. Além disso, também implementamos um código básico para predição de preço de casas. Ao longo deste módulo, implementaremos algumas outras funcionalidades do scikit a fim de melhorar a performance desse modelo.

In [230]:
casas = pandas.read_csv(COMPLEX_DATA_PATH, low_memory=False)
casas = casas.rename(columns={
    "housing_median_age": "mediana_idade_casas",
    "total_rooms": "total_comodos",
    "total_bedrooms": "total_quartos",
    "population": "total_populacao",
    "households": "total_casas",
    "median_income": "mediana_renda_familiar",
    "median_house_value": "mediana_valor_casa",
    "ocean_proximity": "proximidade_oceano"
})

NUMERICAL_FEATURES = ["latitude", "longitude", "mediana_idade_casas", "total_comodos", "total_quartos", "total_populacao", "total_casas", "mediana_renda_familiar"]
CATEGORICAL_FEATURES = ["proximidade_oceano"]
TARGET_FEATURE = "mediana_valor_casa"

X = casas[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y = casas[TARGET_FEATURE]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, NUMERICAL_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

full_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", LinearRegression())
    ]
)

full_pipeline.fit(X_train, y_train)

print(f"Average R2 Score: {cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring="r2").mean()*100:.3f}%")
print(f"Standard Deviation: {cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring="r2").std()*100:.3f}%")

Average R2 Score: 64.754%
Standard Deviation: 1.145%


### Trindade do Scikit Learn

Os Estimators são a base de tudo no Scikit Learn. Até mesmo os Transformers são Estimators e portanto podem aprender parâmetros a partir da matriz de características (X). Todo recurso no Scikit Learn que tenha um método fit é um Estimator, todo recurso que tenha um método transform é um Transformer e todo recurso que tenha um método predict é um Predictor.

| Classe | Método Principal | Função |
| --- | --- | --- |
| Estimator | .fit() | Aprende parâmetros a partir da matriz de características |
| Transformer | .transform() | Modifica a matriz de características limpando, escalando, criando colunas, etc. |
| Predictor | .predict() | Gera a resposta final com base no que aprendeu |

### FunctionTransformer

Imagine que queremos fazer um cálculo customizado com base em outras colunas do dataset, podendo incluir essa parte do processamento na pipeline ao invés de isolá-la no pandas. Para isso, podemos usar a classe FunctionTransformer do scikit. Para poder utilizar um objeto como transformer (o segundo valor em cada step das pipelines), o objeto tem de ter um método fit e um método transform. Quando não precisamos de um método fit para que o transformer aprenda algum padrão com base nos dados, podemos utilizar a classe FunctionTransformer. Para isso, a classe deve receber uma função que realize o cálculo.

Abaixo, definimos outra lista de features. Nela, adicionamos duas features que vamos gerar com essa nova funcionalidade e removemos as antigas que não eram mais necessárias. Depois, definimos uma função que recebe um dataframe e retorna um dataframe. Essa função tem de fazer uma cópia do dataframe original para mantê-lo intácto. Após copiar o dataframe, podemos realizar nossos cálculos como faríamos no pandas naturalmente. Por fim, retornamos o novo dataframe com as novas colunas que calculamos. 

Para implantar esse novo transformer na pipeline do modelo, basta adicionar mais uma tupla aos steps da pipeline. Na grande maioria dos casos, o passo de feature engineering deve vir antes do pré-processamento para evitar que o passo de feature engineering calcule dados completamente fora do padrão. Assim, se uma das colunas do cálculo for nula, o resultado será nulo. E se o resultado for nulo, ele será preenchido pelo imputador utilizando a regra que definimos. Nesse caso, a mediana.

Perceba que essa pequena modificação já aumentou a performance do nosso modelo. Antes, o R2 cruzado era de 63.572% e agora é de 64.326%. Um aumento que parece insignificante, mas não é. Além de aumentar o R2 cruzado, o desvio padrão se mantede quase intocado. Com isso, tanto a banda inferior como a banda superior do modelo refinado pelo feature engineering são mais favoráveis do que no modelo anterior. No mundo real, vence o modelo que tenha a melhor métrica, desde que o ganho justifique a complexidade que a solução adiciona ao sistema.

In [231]:
from sklearn.preprocessing import FunctionTransformer

NUMERICAL_FEATURES_ENGINEERED = NUMERICAL_FEATURES.copy()
NUMERICAL_FEATURES_ENGINEERED.append("quartos_por_comodo")
NUMERICAL_FEATURES_ENGINEERED.append("comodos_por_casa")
NUMERICAL_FEATURES_ENGINEERED.append("quartos_por_casa")
NUMERICAL_FEATURES_ENGINEERED.remove("total_quartos")
NUMERICAL_FEATURES_ENGINEERED.remove("total_comodos")

def feature_engineering(dataframe: pandas.DataFrame) -> pandas.DataFrame:
    dataframe = dataframe.copy()

    dataframe["quartos_por_comodo"] = dataframe["total_quartos"] / dataframe["total_comodos"]
    dataframe["comodos_por_casa"] = dataframe["total_comodos"] / dataframe["total_casas"]
    dataframe["quartos_por_casa"] = dataframe["total_quartos"] / dataframe["total_casas"]
    dataframe["mediana_renda_familiar"] = dataframe["mediana_renda_familiar"] * 10000

    return dataframe

FeatureTransformer = FunctionTransformer(feature_engineering)

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, NUMERICAL_FEATURES_ENGINEERED),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

full_pipeline = Pipeline(
    steps=[
        ("feature_engineering", FeatureTransformer),
        ("preprocessing", preprocessor),
        ("model", LinearRegression())
    ]
)

full_pipeline.fit(X_train, y_train)

print(f"Average R2 Score: {cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring="r2").mean()*100:.3f}%")
print(f"Standard Deviation: {cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring="r2").std()*100:.3f}%")

Average R2 Score: 65.579%
Standard Deviation: 1.184%


Se você quiser verificar se o transformer realizou o trabalho conforme o que você desejava, basta utilizar o método transform dele em um dataframe que tenha todas as colunas necessárias. Dessa forma, o dataframe de resultado terá as novas colunas que o transformer calcula.

In [232]:
print(FeatureTransformer.transform(X_test))

       latitude  longitude  ...  comodos_por_casa  quartos_por_casa
20046     36.06    -119.01  ...          4.192201               NaN
3024      35.14    -119.46  ...          5.039384               NaN
15663     37.80    -122.44  ...          3.977155               NaN
20484     34.28    -118.72  ...          6.163636               NaN
9814      36.62    -121.93  ...          5.492991               NaN
...         ...        ...  ...               ...               ...
15362     33.36    -117.22  ...          7.002212          1.066372
16623     35.36    -120.83  ...          6.131915          1.256738
18086     37.31    -122.05  ...          7.237676          0.947183
2144      36.77    -119.76  ...          5.289030          0.983122
3665      34.22    -118.37  ...          3.988839          1.033482

[4128 rows x 12 columns]


Como você deve ter imaginado, o R2 Score para esse modelo não foi o mais eficiente possível. O motivo para isso é muito simples: O algoritmo que usamos não é o mais adequado para esse caso. Para resolver isso, basta trocar o algoritmo usado por um mais apropriado e o R2 Score pode ultrapassar os 80%.